In [1]:
import pandas as pd
from joblib import load

# 1. 모델 & 벡터라이저 로드 (email_phising/models)
model_path = "../models/logistic_eda_model.joblib"
vectorizer_path = "../models/tfidf_vectorizer.joblib"

model_eda = load(model_path)
tfidf_vectorizer = load(vectorizer_path)

# 2. 데이터 로드 (email_phising/data)
#   - 예시: CEAS_08.csv, 컬럼: [body, urls, label(선택적)]
df = pd.read_csv("../data/CEAS_08.csv")

# 3. TF-IDF 변환
X_tfidf = tfidf_vectorizer.transform(df["body"].astype(str))

# 4. TF-IDF를 DataFrame으로 변환 (+ feature 이름 부여)
X = pd.DataFrame(
    X_tfidf.toarray(), columns=tfidf_vectorizer.get_feature_names_out()
)

# 5. urls 특성 추가 (데이터에 있는 경우)
if "urls" in df.columns:
    X["urls"] = df["urls"]

# 6. label 컬럼은 feature에서 제거 (있을 수도 있음)
if "label" in X.columns:
    X = X.drop(columns=["label"])

# 7. 모델이 학습할 때 사용한 feature 순서/이름에 정확히 맞추기
if hasattr(model_eda, "feature_names_in_"):
    X = X[model_eda.feature_names_in_]

print("X shape:", X.shape)
print("예시 feature 컬럼 몇 개:", list(X.columns[:10]))


ModuleNotFoundError: No module named 'scipy.sparse._csr'

In [ ]:
import numpy as np
import pandas as pd

# 로지스틱 회귀 계수 가져오기
coef = model_eda.coef_[0]   # 이진 분류라 클래스 하나

# X의 컬럼 이름 (TF-IDF 단어 + urls 등 메타특징)
feature_names = X.columns

coef_df = pd.DataFrame({
    'feature': feature_names,
    'coef': coef
})

# 피싱(1) 쪽으로 강하게 밀어주는 단어/특징
top_phishing = coef_df.sort_values('coef', ascending=False).head(20)

# 정상(0) 쪽으로 강하게 밀어주는 단어/특징
top_legit = coef_df.sort_values('coef', ascending=True).head(20)

print("=== 피싱(1) 쪽으로 강하게 기여하는 상위 20개 특징 ===")
print(top_phishing)

print("\n=== 정상(0) 쪽으로 강하게 기여하는 상위 20개 특징 ===")
print(top_legit)


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# coef_df = 회귀 계수를 담은 데이터프레임
coef = model_eda.coef_[0]
feature_names = X.columns

coef_df = pd.DataFrame({
    'feature': feature_names,
    'coef': coef
})

# 피싱(+), 정상(-) 기여 상위 특징
top_pos = coef_df.sort_values("coef", ascending=False).head(20)
top_neg = coef_df.sort_values("coef", ascending=True).head(20)

def plot_coefs(df, title, color):
    plt.figure(figsize=(8, 6))
    df_sorted = df.sort_values("coef")
    plt.barh(df_sorted["feature"], df_sorted["coef"], color=color)
    plt.title(title)
    plt.xlabel("Coefficient")
    plt.tight_layout()
    plt.show()

plot_coefs(top_pos, "Top 20 Features Driving Phishing (Positive Coefficients)", "red")
plot_coefs(top_neg, "Top 20 Features Driving Legitimate Emails (Negative Coefficients)", "blue")
